# Katyayani — root-cause ranking on real fault data

Model 6 of the nine Sati models (`docs/adr/Navadurga.md`). Given an incident's metrics, rank the services by how likely each is the root cause.

**Data:** RCAEval (Pham et al., WWW 2025) — real fault injections on Online Boutique, Sock Shop and Train Ticket.
**Method:** per-service robust shift / peak / onset per metric family, log-compressed and ranked within the incident; LightGBM `lambdarank`.
**Honest test:** leave one *system* out — the model is scored on a system it never saw.

The committed results table is `docs/research/models/sati-rca.md`; this notebook reproduces it step by step.

In [ ]:
# Run on Kaggle or Colab. Clones this repository and installs the two packages.
import os, subprocess, sys
if not os.path.exists("PASHUPATASTRA"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/preetraval45/PASHUPATASTRA.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "PASHUPATASTRA/packages/core", "-e", "PASHUPATASTRA/packages/sati", "matplotlib"], check=True)
sys.path[:0] = ["PASHUPATASTRA/packages/core", "PASHUPATASTRA/packages/sati"]

## 1. Load the case index and one incident

In [ ]:
from pathlib import Path
from sati.sources import rcaeval
cache = Path("cache/rcaeval")
rev = rcaeval.revision()
cases = rcaeval.cases(cache, rev)
print(rev[:12], cases.shape)
cases.groupby(["dataset", "fault"]).size().unstack(fill_value=0)

In [ ]:
case = cases.iloc[0]
frame = rcaeval.metrics(case.case, cache, rev)
print(case.case, "root cause:", case.root_cause_service, "fault:", case.fault)
frame.filter(like=case.root_cause_service).plot(x=None, figsize=(10, 3), legend=False, title="the root cause's metrics around the injection");

## 2. Features — what changed, relative to each series' own normal

In [ ]:
from sati.rca import features as feats
table, provenance = feats.incident_features(frame, int(case.inject_time))
table.sort_values("shift_any", ascending=False).head(8)[["service", "shift_any", "peak_any", "onset_any", "rank_shift_any"]]

## 3. Train and score with leave-one-system-out (the full run)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "sati.rca.run", "--cache", "cache/rcaeval", "--out", "results/katyayani"], check=True, cwd="PASHUPATASTRA/packages/sati")

In [ ]:
import json, pandas as pd
r = json.load(open("PASHUPATASTRA/packages/sati/results/katyayani/results.json"))
pd.DataFrame([{"split": s["scheme"], "held out": s["fold"], "Katyayani AC@1": s["sati"]["ac@1"], "baseline AC@1": s["baseline"]["ac@1"]} for s in r["splits"]])

## 4. What a ranking cites

Every feature records the metric series it was computed from — rule 1 in `CLAUDE.md`.

In [ ]:
r["example_explanation"]